# Notebook 3: Complete Business Analysis Project

In this notebook, we'll apply everything you've learned to conduct a complete business analysis.

## Project Scenario

You are a business analyst at a retail company. Management has asked you to:

1. **Analyze sales performance** across products, regions, and time
2. **Identify trends** and patterns in the data
3. **Make recommendations** for business decisions
4. **Create visualizations** for presentation to stakeholders

Let's build a comprehensive analysis report!

## Step 1: Setup and Data Loading

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set styles for better looking charts
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("✅ Libraries loaded successfully!")

In [ ]:
# Load the sales data
df = pd.read_csv('../data/superstore_sales.csv')

# Convert date to datetime
df['Date'] = pd.to_datetime(df['Date'])

print(f"✅ Data loaded: {df.shape[0]} transactions")
print(f"Date range: {df['Date'].min().date()} to {df['Date'].max().date()}")

## Step 2: Data Preparation and Feature Engineering

In [ ]:
# Calculate business metrics
df['Profit'] = df['Sales'] - df['Cost']
df['Profit_Margin'] = (df['Profit'] / df['Sales']) * 100
df['Revenue_Per_Unit'] = df['Sales']  # Assuming 1 unit per transaction

# Extract date components
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Month_Name'] = df['Date'].dt.month_name()
df['Quarter'] = df['Date'].dt.quarter
df['Week'] = df['Date'].dt.isocalendar().week
df['Day_of_Week'] = df['Date'].dt.day_name()

# Add product categories
product_info = pd.DataFrame({
    'Product': ['Laptop', 'Mouse', 'Keyboard', 'Monitor'],
    'Category': ['Electronics', 'Accessories', 'Accessories', 'Electronics'],
    'Supplier': ['TechCorp', 'PeripheralPlus', 'PeripheralPlus', 'ScreenPro'],
    'Target_Margin': [35, 65, 55, 40]  # Target profit margin %
})

df = df.merge(product_info, on='Product', how='left')

# Calculate performance vs target
df['Margin_vs_Target'] = df['Profit_Margin'] - df['Target_Margin']
df['Meeting_Target'] = df['Margin_vs_Target'] >= 0

print("✅ Data preparation complete!")
df.head()

## Step 3: Executive Summary Dashboard

### Key Performance Indicators (KPIs)

In [ ]:
# Calculate KPIs
total_sales = df['Sales'].sum()
total_profit = df['Profit'].sum()
avg_profit_margin = df['Profit_Margin'].mean()
total_transactions = len(df)
avg_transaction_value = df['Sales'].mean()
target_achievement = (df['Meeting_Target'].sum() / len(df)) * 100

print("="*60)
print("EXECUTIVE SUMMARY - KEY PERFORMANCE INDICATORS")
print("="*60)
print(f"\n📊 Total Sales Revenue:        ${total_sales:,.2f}")
print(f"💰 Total Profit:               ${total_profit:,.2f}")
print(f"📈 Average Profit Margin:      {avg_profit_margin:.2f}%")
print(f"🛒 Total Transactions:         {total_transactions:,}")
print(f"💵 Average Transaction Value:  ${avg_transaction_value:,.2f}")
print(f"🎯 Target Achievement Rate:    {target_achievement:.1f}%")
print("="*60)

## Step 4: Product Performance Analysis

In [ ]:
# Product analysis
product_performance = df.groupby('Product').agg({
    'Sales': ['sum', 'mean', 'count'],
    'Profit': ['sum', 'mean'],
    'Profit_Margin': 'mean',
    'Meeting_Target': lambda x: (x.sum() / len(x)) * 100
}).round(2)

product_performance.columns = ['Total_Sales', 'Avg_Sale', 'Count', 
                                'Total_Profit', 'Avg_Profit', 
                                'Avg_Margin', 'Target_Achievement_%']

product_performance = product_performance.sort_values('Total_Sales', ascending=False)

print("\nProduct Performance Summary:")
product_performance

In [ ]:
# Visualize product performance
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Sales by product
product_sales = df.groupby('Product')['Sales'].sum().sort_values(ascending=False)
axes[0, 0].bar(product_sales.index, product_sales.values, color='steelblue')
axes[0, 0].set_title('Total Sales by Product', fontweight='bold', fontsize=12)
axes[0, 0].set_ylabel('Sales ($)')
axes[0, 0].tick_params(axis='x', rotation=45)

# Profit by product
product_profit = df.groupby('Product')['Profit'].sum().sort_values(ascending=False)
axes[0, 1].bar(product_profit.index, product_profit.values, color='green')
axes[0, 1].set_title('Total Profit by Product', fontweight='bold', fontsize=12)
axes[0, 1].set_ylabel('Profit ($)')
axes[0, 1].tick_params(axis='x', rotation=45)

# Profit margin by product
product_margin = df.groupby('Product')['Profit_Margin'].mean().sort_values(ascending=False)
colors = ['green' if x >= 40 else 'orange' for x in product_margin.values]
axes[1, 0].bar(product_margin.index, product_margin.values, color=colors)
axes[1, 0].axhline(y=40, color='red', linestyle='--', label='Target: 40%')
axes[1, 0].set_title('Average Profit Margin by Product', fontweight='bold', fontsize=12)
axes[1, 0].set_ylabel('Profit Margin (%)')
axes[1, 0].tick_params(axis='x', rotation=45)
axes[1, 0].legend()

# Transaction count by product
product_count = df.groupby('Product').size().sort_values(ascending=False)
axes[1, 1].bar(product_count.index, product_count.values, color='coral')
axes[1, 1].set_title('Transaction Count by Product', fontweight='bold', fontsize=12)
axes[1, 1].set_ylabel('Number of Transactions')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## Step 5: Regional Performance Analysis

In [ ]:
# Regional analysis
regional_performance = df.groupby('Region').agg({
    'Sales': 'sum',
    'Profit': 'sum',
    'Profit_Margin': 'mean',
    'Product': 'count'
}).round(2)

regional_performance.columns = ['Total_Sales', 'Total_Profit', 'Avg_Margin', 'Transactions']
regional_performance['Sales_Share_%'] = (regional_performance['Total_Sales'] / 
                                          regional_performance['Total_Sales'].sum() * 100).round(2)

print("\nRegional Performance Summary:")
regional_performance.sort_values('Total_Sales', ascending=False)

In [ ]:
# Regional visualizations
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sales by region - pie chart
region_sales = df.groupby('Region')['Sales'].sum()
axes[0].pie(region_sales, labels=region_sales.index, autopct='%1.1f%%', startangle=90)
axes[0].set_title('Sales Distribution by Region', fontweight='bold', fontsize=12)

# Profit by region - bar chart
region_profit = df.groupby('Region')['Profit'].sum().sort_values(ascending=False)
axes[1].bar(region_profit.index, region_profit.values, color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A'])
axes[1].set_title('Total Profit by Region', fontweight='bold', fontsize=12)
axes[1].set_ylabel('Profit ($)')

plt.tight_layout()
plt.show()

## Step 6: Time Series Analysis

In [ ]:
# Monthly trends
monthly_trends = df.groupby('Month_Name').agg({
    'Sales': 'sum',
    'Profit': 'sum',
    'Product': 'count'
}).reindex(['January', 'February', 'March', 'April', 'May', 'June'])

monthly_trends.columns = ['Sales', 'Profit', 'Transactions']
monthly_trends['Profit_Margin_%'] = (monthly_trends['Profit'] / monthly_trends['Sales'] * 100).round(2)

print("\nMonthly Performance Trends:")
monthly_trends

In [ ]:
# Time series visualization
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Monthly sales and profit trend
months = ['January', 'February', 'March', 'April', 'May', 'June']
x = range(len(months))

ax1 = axes[0]
ax2 = ax1.twinx()

ax1.plot(x, monthly_trends['Sales'], marker='o', linewidth=2, markersize=8, 
         color='steelblue', label='Sales')
ax2.plot(x, monthly_trends['Profit'], marker='s', linewidth=2, markersize=8, 
         color='green', label='Profit')

ax1.set_xlabel('Month', fontsize=11)
ax1.set_ylabel('Sales ($)', color='steelblue', fontsize=11)
ax2.set_ylabel('Profit ($)', color='green', fontsize=11)
ax1.set_title('Monthly Sales and Profit Trends', fontweight='bold', fontsize=13)
ax1.set_xticks(x)
ax1.set_xticklabels(months, rotation=45)
ax1.tick_params(axis='y', labelcolor='steelblue')
ax2.tick_params(axis='y', labelcolor='green')
ax1.grid(True, alpha=0.3)
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')

# Quarterly comparison
quarterly_sales = df.groupby('Quarter')['Sales'].sum()
axes[1].bar(quarterly_sales.index, quarterly_sales.values, color='purple', alpha=0.7)
axes[1].set_title('Quarterly Sales Performance', fontweight='bold', fontsize=13)
axes[1].set_xlabel('Quarter', fontsize=11)
axes[1].set_ylabel('Sales ($)', fontsize=11)
axes[1].set_xticks([1, 2])
axes[1].set_xticklabels(['Q1', 'Q2'])
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## Step 7: Cross-Dimensional Analysis (Product × Region)

In [ ]:
# Pivot table: Sales by Product and Region
pivot_sales = pd.pivot_table(
    df,
    values='Sales',
    index='Product',
    columns='Region',
    aggfunc='sum',
    fill_value=0,
    margins=True,
    margins_name='Total'
).round(2)

print("\nSales by Product and Region:")
print(pivot_sales)

# Pivot table: Average Profit Margin by Product and Region
pivot_margin = pd.pivot_table(
    df,
    values='Profit_Margin',
    index='Product',
    columns='Region',
    aggfunc='mean'
).round(2)

print("\nAverage Profit Margin by Product and Region:")
print(pivot_margin)

In [ ]:
# Heatmap visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Sales heatmap
sns.heatmap(pivot_sales.iloc[:-1, :-1], annot=True, fmt='.0f', cmap='YlGnBu', 
            ax=axes[0], cbar_kws={'label': 'Sales ($)'})
axes[0].set_title('Sales Heatmap: Product × Region', fontweight='bold', fontsize=12)

# Profit margin heatmap
sns.heatmap(pivot_margin, annot=True, fmt='.1f', cmap='RdYlGn', 
            ax=axes[1], cbar_kws={'label': 'Profit Margin (%)'})
axes[1].set_title('Profit Margin Heatmap: Product × Region', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.show()

## Step 8: Insights and Recommendations

In [ ]:
# Find top and bottom performers
best_product = product_performance['Total_Sales'].idxmax()
worst_product = product_performance['Total_Sales'].idxmin()

best_region = regional_performance['Total_Sales'].idxmax()
worst_region = regional_performance['Total_Sales'].idxmin()

# Find products below target margin
below_target = df.groupby('Product').agg({
    'Profit_Margin': 'mean',
    'Target_Margin': 'first'
})
below_target['Gap'] = below_target['Profit_Margin'] - below_target['Target_Margin']
underperformers = below_target[below_target['Gap'] < 0].sort_values('Gap')

print("="*70)
print("KEY INSIGHTS AND RECOMMENDATIONS")
print("="*70)

print("\n📊 TOP PERFORMERS:")
print(f"  • Best Product: {best_product} (${product_performance.loc[best_product, 'Total_Sales']:,.2f} in sales)")
print(f"  • Best Region: {best_region} (${regional_performance.loc[best_region, 'Total_Sales']:,.2f} in sales)")

print("\n⚠️  AREAS OF CONCERN:")
if len(underperformers) > 0:
    print("  • Products below target margin:")
    for product, row in underperformers.iterrows():
        print(f"    - {product}: {row['Profit_Margin']:.1f}% vs target {row['Target_Margin']:.1f}%")
else:
    print("  • All products meeting or exceeding target margins! ✅")

print(f"\n  • Lowest performing product: {worst_product}")
print(f"  • Lowest performing region: {worst_region}")

print("\n💡 RECOMMENDATIONS:")
print("\n1. PRODUCT STRATEGY:")
print(f"   • Focus marketing efforts on {best_product} (highest sales)")
if len(underperformers) > 0:
    worst_margin_product = underperformers.index[0]
    print(f"   • Review pricing/costs for {worst_margin_product} (below target margin)")
print(f"   • Consider expanding {best_product} product line")

print("\n2. REGIONAL STRATEGY:")
print(f"   • Allocate more resources to {best_region} region (highest sales)")
print(f"   • Investigate challenges in {worst_region} region")
print(f"   • Consider targeted promotions in underperforming regions")

print("\n3. OPERATIONAL IMPROVEMENTS:")
print("   • Monitor profit margins closely, especially for Laptops")
print("   • Optimize inventory based on regional demand patterns")
print("   • Implement dynamic pricing strategies for seasonal trends")

print("\n4. GROWTH OPPORTUNITIES:")
print("   • Cross-sell accessories with electronics purchases")
print("   • Bundle products to increase average transaction value")
print("   • Launch loyalty program in high-performing regions")

print("\n" + "="*70)

## Step 9: Export Results for Presentation

In [ ]:
# Create summary report
summary_report = {
    'Metric': [
        'Total Sales',
        'Total Profit',
        'Average Profit Margin',
        'Total Transactions',
        'Average Transaction Value',
        'Target Achievement Rate'
    ],
    'Value': [
        f"${total_sales:,.2f}",
        f"${total_profit:,.2f}",
        f"{avg_profit_margin:.2f}%",
        f"{total_transactions:,}",
        f"${avg_transaction_value:,.2f}",
        f"{target_achievement:.1f}%"
    ]
}

summary_df = pd.DataFrame(summary_report)

print("\n📊 EXECUTIVE SUMMARY REPORT")
print(summary_df.to_string(index=False))

# Optionally save to CSV
# summary_df.to_csv('../output/executive_summary.csv', index=False)
# product_performance.to_csv('../output/product_performance.csv')
# regional_performance.to_csv('../output/regional_performance.csv')

## Conclusion

Congratulations! 🎉 You've completed a full business analysis project using Python and Pandas.

### What You've Learned:

1. **Data Loading and Preparation**: Clean and structure data for analysis
2. **Feature Engineering**: Create calculated fields and metrics
3. **Aggregation and Grouping**: Summarize data across multiple dimensions
4. **Pivot Tables**: Create cross-tabulations for deep insights
5. **Data Visualization**: Create professional charts and graphs
6. **Business Intelligence**: Derive actionable insights from data

### Excel vs Python - The Complete Journey:

| Task | Excel Approach | Python Approach | Advantage |
|------|----------------|-----------------|------------|
| Load Data | Manual file open | `pd.read_csv()` | Automated |
| Calculations | Formula in each cell | Vectorized operations | 100x faster |
| Pivot Tables | Manual drag & drop | `groupby()` + `pivot_table()` | More flexible |
| Charts | Insert chart wizard | `matplotlib` / `seaborn` | More customizable |
| Analysis | Manual interpretation | Automated insights | Reproducible |
| Reporting | Manual copy-paste | Automated export | Time-saving |

### Next Steps:

- Apply these techniques to your own business data
- Explore more advanced topics: machine learning, forecasting, optimization
- Build automated reporting dashboards
- Share your analyses with stakeholders

**Remember**: The power of Python isn't just in doing what Excel does faster—it's in doing things Excel can't do at all! 🚀